# Stimulus overview

One figure describing the stimulus set the coherence analysis runs on: how much
listening time each recording contributes, the rates of its linguistic units, and
its modulation and pitch spectra.

Inputs come from `speech_units_dir` (the `scripts/speech_units_pitch/` outputs),
`audio_features_dir` (gammatone envelopes) and `coherence_dir` (per-trial files,
used only to count listening time per stimulus).

In [ ]:
import colorsys
import json
import re
from pathlib import Path

import csv
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor

import mne

import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from scipy.ndimage import median_filter
from scipy.signal import welch
from scipy.stats import false_discovery_control, pearsonr

ROOT = Path.cwd().parent if Path.cwd().name == "figures" else Path.cwd()
CONFIG = json.loads((ROOT / "src" / "package_data" / "config.json").read_text())

SPEECH_UNITS_DIR = Path(CONFIG["speech_units_dir"])
FEATURES_DIR = Path(CONFIG["audio_features_dir"])
COHERENCE_DIR = Path(CONFIG["coherence_dir"])
PITCH_DIR = SPEECH_UNITS_DIR / "pitch"

# Envelopes are written at the audio_features factory's target_fs.
ENVELOPE_FS = 1024.0

# --- house style -------------------------------------------------------------
# Font scale and figure size taken from the original figure notebook.
RATE_X_FONTSIZE = 8
RATE_X_ROTATION = 20
DEFAULT_FONTSIZE = 10
TICK_FONTSIZE = 8
SMALL_FONTSIZE = 6.5
PANEL_LABEL_FONTSIZE = 14
FIGSIZE = (6.27, 7 * 8.3 / 8)

INK, MUTED, HAIRLINE = "#1b1b1a", "#6b6b68", "#d5d5d1"
ACCENT = "#c8442e"
# Figure 2's syllable colour, for the places where the syllabic rate is marked as a
# series in its own right -- so the same series reads the same in both notebooks.
# ACCENT stays the chrome colour for the rate boxplots of panels c-e, which cover
# words, phonemes and intonation units as well and are not a syllable marker.
SYLLABLE_COLOUR = "#06d6a0"
NEUTRAL = "#9a9a96"
BAR = "C0"

# Blue-red, as in the original figure. A colourbar carries the scale.
PITCH_GREY = NEUTRAL       # what the middle of the scale fades to
PITCH_GREY_WIDTH = 0.20    # its reach, as a fraction of the scale


def grey_centre(cmap, grey, sigma, n=256, name="pitch"):
    """`cmap` with its pale middle blended into `grey`.

    A diverging map passes through near-white at its midpoint -- RdBu_r reaches a
    lightness of 0.97 -- so the stimuli whose median F0 lands mid-scale come out
    invisible against the page. Blending the centre towards a mid grey gives them
    a weight to read at while keeping the blue-to-red sense of the scale.

    The blend is a Gaussian in scale position, not a lightness cap: capping
    lightness leaves the hue flip at the midpoint intact and merely darkens it
    into view, which puts a visible seam down the middle of the colourbar.
    """
    positions = np.linspace(0, 1, n)
    colours = cmap(positions)
    weight = np.exp(-(((positions - 0.5) / sigma) ** 2))[:, None]
    colours[:, :3] = colours[:, :3] * (1 - weight) + np.array(mcolors.to_rgb(grey)) * weight
    return mcolors.LinearSegmentedColormap.from_list(name, colours)


PITCH_CMAP = grey_centre(plt.get_cmap("RdBu_r"), PITCH_GREY, PITCH_GREY_WIDTH)

plt.rcParams.update({
    "font.size": TICK_FONTSIZE,
    "axes.labelsize": DEFAULT_FONTSIZE,
    "axes.titlesize": DEFAULT_FONTSIZE,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "axes.linewidth": 0.8,
    "figure.dpi": 130,
    # Embed text as TrueType, not as the matplotlib default Type 3. Type 3 fonts
    # are what Elsevier and most publishers flag or reject outright, and text in
    # them is not reliably searchable or selectable in the final PDF. This costs
    # nothing and has to be set before any figure is drawn.
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def bare(ax):
    """Recessive chrome: drop the top/right rules."""
    ax.spines[["top", "right"]].set_visible(False)
    return ax

## Stimulus inventory

Listening time per stimulus, rebuilt from the per-trial coherence filenames rather
than a checked-in table, so it always reflects the data actually on disk. The
exclusions match the coherence pipeline: `sin` and `av` conditions, and the
`_shifted` / `_artefact` recordings.

In [ ]:
TRIAL_PATTERN = re.compile(
    r"sub-(?P<subject>\d+)_-_ses-\d+_-_trial-\d+_-_cond-(?P<cond>[^_]+)"
    r"_-_audio\d+-(?P<audio>.+)_-_eeg_coherence\.npz"
)
EXCLUDED_CONDITIONS = {"sin", "av"}
EXCLUDED_MARKERS = ("_shifted", "_artefact")


def stimulus_group(audio_name):
    """audiobook_10_2 -> 'audiobook 10'; the parts of one recording pool together."""
    kind, number, *_ = audio_name.split("_")
    return f"{kind} {int(number):02d}"


trial_files = sorted(COHERENCE_DIR.glob("*.npz"))

records = []
for path in tqdm(trial_files, desc="coherence trials", unit="file"):
    match = TRIAL_PATTERN.match(path.name)
    if match is None or match["cond"] in EXCLUDED_CONDITIONS:
        continue
    if any(marker in match["audio"] for marker in EXCLUDED_MARKERS):
        continue
    with np.load(path) as trial:
        listened = float(trial["analysed_duration"])
    records.append({
        "audio": match["audio"],
        "group": stimulus_group(match["audio"]),
        "subject": match["subject"],
        "listening_hours": listened / 3600.0,
    })

trials = pd.DataFrame(records)

by_stimulus = trials.groupby("audio").agg(
    listening_hours=("listening_hours", "sum"),
    n_subjects=("subject", "nunique"),
)
by_group = trials.groupby("group").agg(
    listening_hours=("listening_hours", "sum"),
    n_subjects=("subject", "nunique"),
).sort_values("listening_hours")

STIMULI = list(by_stimulus.index)
# Each stimulus weighs on the averages by how much of the dataset it accounts for.
WEIGHTS = by_stimulus.loc[STIMULI, "listening_hours"].to_numpy()

print(f"{len(trials)} trials, {trials.subject.nunique()} subjects, "
      f"{len(STIMULI)} stimuli, {trials.listening_hours.sum():.1f} listening hours")

## Speech unit rates

In [ ]:
def load_units(name):
    """Load one speech-unit JSON, restricted to the analysed stimuli."""
    payload = json.loads((SPEECH_UNITS_DIR / name).read_text())
    return {k: v for k, v in payload.items() if k in by_stimulus.index}


word_rates = load_units("word_rates.json")
syllable_rates_heuristic = load_units("syllabic_rates_heuristic.json")
syllable_rates_dejong = load_units("syllabic_rates_dejong.json")
phoneme_rates = load_units("phoneme_rates.json")
categorical_rates = load_units("categorical_phoneme_rates.json")
iu_rates = load_units("iu_rates.json")
sentence_rates = load_units("sentence_rates.json")
clause_rates = load_units("clause_rates.json")
nuclei_onsets = load_units("nuclei_onsets_heuristic.json")

for label, rates in [
    ("sentences", sentence_rates), ("clauses", clause_rates),
    ("intonation units", iu_rates), ("words", word_rates),
    ("syllables (heuristic)", syllable_rates_heuristic),
    ("syllables (de Jong)", syllable_rates_dejong), ("phonemes", phoneme_rates),
]:
    values = np.array(list(rates.values()))
    print(f"{label:24s} n={len(values):3d}  median {np.median(values):.2f} Hz")

## Modulation spectra and pitch distributions

Welch PSD of each broadband gammatone envelope, and the F0 histograms from
`formants_and_pitch.py`. Both are ordered by the stimulus's median F0, which is
also what colours them.

In [ ]:
psds, pitch_counts = [], []
for name in STIMULI:
    with np.load(FEATURES_DIR / f"{name}.npz") as features:
        envelope = features["gammatone_envelope"].squeeze()
    frequencies, power = welch(
        envelope, fs=ENVELOPE_FS,
        nperseg=int(ENVELOPE_FS * 5), noverlap=int(ENVELOPE_FS * 4),
    )
    psds.append(power)

    with np.load(PITCH_DIR / f"{name}_pitch.npz") as pitch:
        pitch_counts.append(pitch["counts"])
        pitch_bins = pitch["bin_centres"]

psds = np.array(psds)
pitch_counts = np.array(pitch_counts)

# Median F0 per stimulus, from the cumulative histogram.
cumulative = np.cumsum(pitch_counts, axis=1)
median_pitch = pitch_bins[np.argmax(cumulative >= cumulative[:, -1:] / 2, axis=1)]

order = np.argsort(median_pitch)
psds, pitch_counts, median_pitch = psds[order], pitch_counts[order], median_pitch[order]
pitch_weights = WEIGHTS[order]

PITCH_NORM = mcolors.Normalize(vmin=np.floor(median_pitch.min() / 10) * 10,
                               vmax=np.ceil(median_pitch.max() / 10) * 10)
print(f"median F0 spans {median_pitch.min():.0f}-{median_pitch.max():.0f} Hz")

## Syllabic rate band

The 5th-95th percentile of instantaneous syllable rate, measured in 5s windows
stepped by 1s and pooled across stimuli by listening time.

This is a *within*-stimulus spread: how much the rate moves from one 5s window to
the next, pooled over the set. It is reported here as a number and is no longer
drawn -- both figures now box the *across*-stimulus spread instead, one rate per
recording, which is the quantity that varies with the stimulus rather than within
it. The two are easy to confuse, so keep the distinction in mind when reading a
syllabic rate off either panel.

In [ ]:
def windowed_rates(onsets, window_len_s=5, step_s=1):
    """Instantaneous unit rate in overlapping windows."""
    onsets = np.asarray(onsets, dtype=float)
    starts = np.arange(0, onsets[-1] - window_len_s, step_s)
    counts = np.array([
        np.sum((onsets >= start) & (onsets < start + window_len_s)) for start in starts
    ])
    return counts / window_len_s


rate_bins = np.arange(-0.1, 15.2, 0.2)
rate_centres = (rate_bins[1:] + rate_bins[:-1]) / 2


def pooled_rate_histogram(onsets_by_stimulus):
    """Listening-time-weighted histogram of instantaneous rates over the stimuli.

    The same construction figure 2's unit-rate panel uses -- 5 s windows stepped by
    1 s, histogrammed per stimulus and pooled by listening time -- on the same 0.2 Hz
    bins, so a histogram drawn from this and one drawn there are the same quantity.
    """
    histograms, weights = [], []
    for name, onsets in onsets_by_stimulus.items():
        counts, _ = np.histogram(windowed_rates(onsets), bins=rate_bins)
        histograms.append(counts)
        weights.append(by_stimulus.loc[name, "listening_hours"])
    pooled = np.average(histograms, axis=0, weights=weights)
    pooled[0] = 0  # silent windows are not a syllable rate
    return pooled


pooled = pooled_rate_histogram(nuclei_onsets)
pooled_cdf = np.cumsum(pooled) / pooled.sum()
SYLLABIC_BAND = (rate_centres[np.argmax(pooled_cdf >= 0.05)],
                 rate_centres[np.argmax(pooled_cdf >= 0.95)])
print(f"syllabic rate 5-95%: {SYLLABIC_BAND[0]:.1f}-{SYLLABIC_BAND[1]:.1f} Hz "
      f"(heuristic estimator)")

# The de Jong estimator's own histogram, which is what the supplementary panel
# overlays. Figure 2's syllable series is built from `nuclei_onsets_dejong.json`, and
# panel b of that figure carries only the de Jong estimator too, so matching it here
# is what makes the overlay the same curve rather than a near neighbour. SYLLABIC_BAND
# above stays on the heuristic estimator, which is what it has always reported.
nuclei_onsets_dejong = load_units("nuclei_onsets_dejong.json")
pooled_dejong = pooled_rate_histogram(nuclei_onsets_dejong)
dejong_cdf = np.cumsum(pooled_dejong) / pooled_dejong.sum()
print(f"syllabic rate 5-95%: {rate_centres[np.argmax(dejong_cdf >= 0.05)]:.1f}-"
      f"{rate_centres[np.argmax(dejong_cdf >= 0.95)]:.1f} Hz (de Jong), "
      f"mode {rate_centres[np.argmax(pooled_dejong)]:.1f} Hz")

## Panels

Each panel is a function of one axis so the layout stays separate from the drawing.

In [ ]:
def label_panels(fig, labels, dy=0.012, align=()):
    """Bold panel letters, placed in figure coordinates.

    Positioning in axes coordinates puts each letter above its own axes, so
    letters that read as a row sit at different heights whenever their panels do.
    Each entry is `ax: (letter, dx)`, with `dx` in axes widths; every tuple in
    `align` is a group forced to share the highest top in it.
    """
    fig.canvas.draw()  # the layout must settle before the boxes mean anything
    to_figure = fig.transFigure.inverted()
    renderer = fig.canvas.get_renderer()

    # Just above each panel's own axes, which leaves the letter tucked beside the
    # title in the gap above it.
    tops = {ax: ax.get_position().y1 for ax in labels}

    # An aligned group instead shares one height, and has to clear the tallest
    # decorations in it -- a title for most panels, but a twin axis for a) -- so
    # that group is measured from the tight bbox rather than the axes box.
    for group in align:
        tops.update(dict.fromkeys(group, max(
            ax.get_tightbbox(renderer).transformed(to_figure).y1 for ax in group
        )))

    for ax, (letter, dx) in labels.items():
        box = ax.get_position()
        fig.text(box.x0 + dx * box.width, tops[ax] + dy, letter,
                 fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold", va="bottom", ha="left")
    return fig


def strip_and_box(ax, series, positions, width=0.45, jitter=0.16, seed=0):
    """Every stimulus as a dot, with a box for the distribution behind it."""
    rng = np.random.default_rng(seed)
    ax.boxplot(
        series, positions=positions, widths=width, showfliers=False,
        boxprops=dict(color=ACCENT, linewidth=1.2),
        whiskerprops=dict(color=ACCENT, linewidth=1.2),
        capprops=dict(color=ACCENT, linewidth=1.2),
        medianprops=dict(color=ACCENT, linewidth=1.2),
    )
    for position, values in zip(positions, series):
        ax.scatter(
            position + rng.uniform(-jitter, jitter, size=len(values)), values,
            color=INK, marker="o", s=3, alpha=0.6, linewidths=0, zorder=3,
        )
    return bare(ax)


def plot_stimulus_overview(ax):
    """Listening hours per stimulus, over the cumulative share of the dataset."""
    y = np.arange(len(by_group))

    # cumulative fraction on the main axis, behind the bars
    share = by_group["listening_hours"].cumsum() / by_group["listening_hours"].sum()
    ax.barh(y, share.values, height=1, alpha=0.3, color="grey")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Cumulative fraction of dataset", labelpad=-0.05)
    ax.set_xticks(np.arange(0, 1.01, 0.2), np.round(np.arange(0, 1.01, 0.2), decimals=1),
                  fontsize=TICK_FONTSIZE)
    ax.set_yticks(y)
    ax.set_yticklabels(by_group.index, fontsize=SMALL_FONTSIZE)

    # listening hours on a twin x-axis
    ax2 = ax.twiny()
    ax2.barh(y, by_group["listening_hours"], height=0.6)
    hours_limit = int(np.ceil(by_group["listening_hours"].max() / 5) * 5) + 5
    ax2.set_xlim(0, hours_limit)
    ax2.set_xlabel("Total listening hours")
    ax2.set_xticks(range(0, hours_limit + 1, 5), range(0, hours_limit + 1, 5),
                   fontsize=TICK_FONTSIZE)

    for each_ax in (ax, ax2):
        each_ax.set_ylim(-1, len(by_group) + 0.5)

    for i, row in enumerate(by_group.itertuples()):
        ax2.text(row.listening_hours + 0.5, i, f"$n={int(row.n_subjects)}$",
                 va="center", ha="left", fontsize=SMALL_FONTSIZE)

    ax2.legend(
        handles=[Rectangle((0, 0), 1, 1, facecolor="grey", alpha=0.3, label="Cumulative fraction"),
                 Rectangle((0, 0), 1, 1, facecolor=BAR, label="Listening hours")],
        loc=(0.31, 0.04), frameon=False, fontsize=TICK_FONTSIZE,
        handlelength=1.5, handleheight=0.6, labelspacing=0.3, borderpad=0.2,
    )
    return ax


def plot_phrase_rates(ax):
    series = [list(sentence_rates.values()),
              list(clause_rates.values()),
              list(iu_rates.values())]
    strip_and_box(ax, series, [0, 1, 2], seed=1)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["Sentences", "Clauses", "Intonational\nunits"], fontsize=RATE_X_FONTSIZE)
    ax.set_yticks([0.2, 0.4, 0.6], ["0.2", "0.4", "0.6"], fontsize=RATE_X_FONTSIZE)
    ax.set_ylabel("Rate (Hz)")
    ax.set_title("Sentence, clause, and IU rates", fontsize=DEFAULT_FONTSIZE)
    return ax


def plot_word_syllable_rates(ax):
    series = [list(word_rates.values()),
              list(syllable_rates_heuristic.values()),
              list(syllable_rates_dejong.values())]
    strip_and_box(ax, series, [0, 1, 2], seed=2)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["Words", "Syllables\n(heuristic)", "Syllables\n(de Jong)"],
                       fontsize=RATE_X_FONTSIZE)
    # One decimal, to give the same tick width as panel b above it.
    ax.set_yticks([2, 3, 4, 5], ["2.0", "3.0", "4.0", "5.0"], fontsize=RATE_X_FONTSIZE)
    ax.set_ylabel("Rate (Hz)")
    ax.set_title("Rates of words and syllables", fontsize=DEFAULT_FONTSIZE)
    return ax


def plot_phoneme_rates(ax):
    categories = ["fricatives", "plosives", "vowels", "consonants", "sonorants"]
    series = [[rates[c] for rates in categorical_rates.values()] for c in categories]
    series.append(list(phoneme_rates.values()))
    strip_and_box(ax, series, list(range(len(series))), width=0.55, jitter=0.22, seed=3)
    ax.set_xticks(range(len(series)))
    ax.set_xticklabels(categories + ["phonemes"], fontsize=RATE_X_FONTSIZE,
                       rotation=RATE_X_ROTATION)
    ax.set_yticks([2, 4, 6, 8, 10, 12], [2, 4, 6, 8, 10, 12], fontsize=RATE_X_FONTSIZE)
    ax.set_ylabel("Rate (Hz)")
    ax.set_title("Phonemic rates", fontsize=DEFAULT_FONTSIZE)
    return ax


def plot_modulation_spectra(ax):
    for power, pitch in zip(psds, median_pitch):
        ax.plot(frequencies, 20 * np.log10(power),
                color=PITCH_CMAP(PITCH_NORM(pitch)), lw=0.4, alpha=0.85)
    ax.set_xlim(0, 400)
    ax.set_yticks([])
    ax.set_xticks(range(0, 401, 50))
    ax.tick_params(axis="x", labelbottom=False)  # panel f below carries the axis
    ax.set_ylabel("Power (dB)")
    ax.set_title("Modulation spectra of stimuli", fontsize=DEFAULT_FONTSIZE)
    return bare(ax)


def plot_pitch_distributions(ax):
    for counts, pitch in zip(pitch_counts, median_pitch):
        density = counts / counts.sum() * 1000
        density[(pitch_bins < 60) | (pitch_bins > 350)] = 0
        ax.plot(pitch_bins, median_filter(density, size=10),
                color=PITCH_CMAP(PITCH_NORM(pitch)), lw=0.5, alpha=0.85)
    ax.set_xlim(0, 400)
    ax.set_yticks([])
    ax.set_xticks(range(0, 401, 50), range(0, 401, 50), fontsize=TICK_FONTSIZE)
    ax.set_xlabel("Frequency (Hz)", fontsize=DEFAULT_FONTSIZE)
    ax.set_ylabel("Probability\ndensity")
    ax.set_title("Histograms of stimulus pitch", fontsize=DEFAULT_FONTSIZE)
    return bare(ax)


def plot_low_frequency_spectra(ax):
    individual = ax.plot(frequencies, psds.T, lw=0.5, c=NEUTRAL, alpha=0.45)
    average = ax.plot(frequencies, np.average(psds, axis=0, weights=pitch_weights),
                      c=INK, lw=1.8)

    ax.set_xlim(0, 15)
    bottom, top = ax.get_ylim()

    # The syllable rate of each stimulus, drawn on the spectra's own frequency
    # axis so the two can be read against each other directly. These are the
    # values panel c shows as "Syllables (de Jong)"; the jitter is vertical
    # only, and carries no meaning beyond separating the crosses.
    rates = np.array([syllable_rates_dejong[name] for name in STIMULI])
    centre, height = top * 0.70, top * 0.10
    ax.boxplot(
        rates, positions=[centre], widths=height, orientation="horizontal",
        showfliers=False, manage_ticks=False, zorder=3,
        boxprops=dict(color=SYLLABLE_COLOUR, linewidth=1.0),
        whiskerprops=dict(color=SYLLABLE_COLOUR, linewidth=1.0),
        capprops=dict(color=SYLLABLE_COLOUR, linewidth=1.0),
        medianprops=dict(color=SYLLABLE_COLOUR, linewidth=1.0),
    )
    rng = np.random.default_rng(4)
    # ax.scatter(rates, centre + rng.uniform(-0.22 * height, 0.22 * height, size=rates.size),
    #            marker="x", s=4, linewidths=0.5, color=SYLLABLE_COLOUR, alpha=0.60, zorder=2)
    ax.text(np.median(rates), centre + 0.90 * height, "Syllabic rate",
            ha="center", va="bottom", fontsize=TICK_FONTSIZE, color=SYLLABLE_COLOUR)
    ax.set_ylim(bottom, top)

    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power (a.u.)")
    ax.set_yticks([])
    ax.set_xticks(range(0, 16, 5), range(0, 16, 5), fontsize=TICK_FONTSIZE)
    ax.set_title("Modulation spectra (low freqs)", fontsize=DEFAULT_FONTSIZE)
    # Legend swatches at a single weight: the plotted lines keep their own widths,
    # but in the legend the difference is identity, not magnitude.
    swatches = [Line2D([], [], color=NEUTRAL, lw=1.5),
                Line2D([], [], color=INK, lw=1.5)]
    ax.legend(swatches, ["Individual stimuli", "Weighted average"],
              loc=(0.35, 0.175), frameon=False, fontsize=TICK_FONTSIZE)
    return bare(ax)

## Figure

In [ ]:
fig = plt.figure(figsize=FIGSIZE, tight_layout=True)
gs0 = gridspec.GridSpec(1, 2, figure=fig)

# --- LEFT COLUMN: panel a on top, the two spectra panels below -----------------
gs_left = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs0[0],
                                           height_ratios=[8, 4], hspace=0.23)
gs_left_bottom = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_left[1],
                                                  height_ratios=[3, 3], hspace=0.4)
ax_overview = fig.add_subplot(gs_left[0])

# a slim column on the right of the spectra pair carries the colour scale
gs_modulation = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs_left_bottom[0],
                                                 width_ratios=[38, 1], wspace=0.06)
gs_pitch = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs_left_bottom[1],
                                            width_ratios=[38, 1], wspace=0.06)
ax_modulation = fig.add_subplot(gs_modulation[0])
ax_pitch = fig.add_subplot(gs_pitch[0])
ax_colourbar = fig.add_subplot(gs_modulation[1])
fig.add_subplot(gs_pitch[1]).set_axis_off()

# --- RIGHT COLUMN: the three rate panels, then the low-frequency spectra --------
gs_right = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs0[1],
                                            height_ratios=[12, 4], hspace=0.22)
gs_right_top = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=gs_right[0], hspace=0.72)
ax_phrases = fig.add_subplot(gs_right_top[0])
ax_words = fig.add_subplot(gs_right_top[1])
ax_phonemes = fig.add_subplot(gs_right_top[2])
ax_lowfreq = fig.add_subplot(gs_right[1])

plot_stimulus_overview(ax_overview)
plot_modulation_spectra(ax_modulation)
plot_pitch_distributions(ax_pitch)
plot_phrase_rates(ax_phrases)
plot_word_syllable_rates(ax_words)
plot_phoneme_rates(ax_phonemes)
plot_low_frequency_spectra(ax_lowfreq)

colourbar = fig.colorbar(
    plt.cm.ScalarMappable(norm=PITCH_NORM, cmap=PITCH_CMAP), cax=ax_colourbar,
)
colourbar.set_label("Median $F_0$ (Hz)", fontsize=SMALL_FONTSIZE)
colourbar.ax.tick_params(labelsize=SMALL_FONTSIZE)
colourbar.outline.set_visible(False)

# The three rate panels carry the same "Rate (Hz)" label over tick columns of
# different widths, so their labels are aligned rather than left where they fall.
fig.align_ylabels([ax_phrases, ax_words, ax_phonemes])

# a) sits over a wider panel, so it needs less of a nudge to clear its ticks.
# a) and b) head the two columns and are placed at a shared height.
label_panels(
    fig,
    {ax_overview: ("a)", -0.12), ax_phrases: ("b)", -0.17), ax_words: ("c)", -0.17),
     ax_phonemes: ("d)", -0.17), ax_modulation: ("e)", -0.17), ax_pitch: ("f)", -0.17),
     ax_lowfreq: ("g)", -0.17)},
    align=[(ax_overview, ax_phrases)],
)

fig.savefig(ROOT / "figures" / "stimulus_overview.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(ROOT / "figures" / "stimulus_overview.png", bbox_inches="tight",
            pad_inches=0.3, dpi=200)
plt.show()

## All trials, and why each is excluded

One row per listening run in the downloaded dataset's `events.tsv` files — the
record of what was actually presented, which is the only complete denominator.
Scanning for EEG files instead would miss runs that were presented but never
recorded.

Each row carries `excluded` and `exclusion_reason`, plus `subject` and `stimulus`
to group on. Exclusions come from two stages and four causes, applied in order:

1. the run was presented but has no EEG file in the download,
2. the EEG exists but never became a standardised trial (could not be aligned),
3. the trial's *condition* is `sin` or `av`,
4. the *recording* is a `_shifted` control or an `_artefact` take.

In [ ]:
DOWNLOADED_DIR = Path(CONFIG["sparrkulee_dir"]).expanduser() / "downloaded"
EEG_DIR = Path(CONFIG["eeg_dir"]).expanduser()

NOT_RECORDED = "presented but no EEG file in the download"
NOT_STANDARDISED = "EEG recorded but not standardised (could not be aligned)"
CONDITION_REASONS = {
    "sin": "speech-in-noise condition",
    "av": "audiovisual condition",
}
RECORDING_REASONS = {
    "_shifted": "pitch-shifted recording",
    "_artefact": "artefact recording",
}

RUN_PATTERN = re.compile(r"sub-(?P<subject>\d+)_ses-(?P<session>[^_]+)_task-\w+_run-(?P<run>\d+)_events\.tsv")
FLAT_PATTERN = re.compile(
    r"sub-(?P<subject>\d+)_-_ses-\d+_-_trial-\d+_-_cond-(?P<cond>[^_]+)_-_audio\d+-(?P<audio>.+)_-_eeg\.fif"
)

# --- what was presented -------------------------------------------------------
event_files = sorted(DOWNLOADED_DIR.glob("sub-*/ses-*/eeg/*_events.tsv"))

run_rows, resting_runs = [], 0
for events_path in tqdm(event_files, desc="events.tsv", unit="run"):
    if "task-listeningActive" not in events_path.name:
        resting_runs += 1
        continue
    match = RUN_PATTERN.match(events_path.name)
    if match is None:
        continue
    with events_path.open(newline="") as handle:
        event = next(iter(csv.DictReader(handle, delimiter="\t")), None)
    stim_file = (event or {}).get("stim_file", "")
    if not stim_file or stim_file == "n/a":
        continue
    run_rows.append({
        "subject": match["subject"],
        "session": match["session"],
        "run": int(match["run"]),
        "stimulus": Path(stim_file).name.replace(".npz.gz", ""),
        "hours": float(event["duration"]) / 3600.0,
        "has_eeg": Path(str(events_path).replace("_events.tsv", "_eeg.bdf")).exists(),
    })

trials = pd.DataFrame(run_rows).sort_values(["subject", "run"]).reset_index(drop=True)

# --- what survived standardisation -------------------------------------------
standardised = defaultdict(list)
for path in EEG_DIR.glob("*.fif"):
    match = FLAT_PATTERN.match(path.name)
    if match:
        standardised[(match["subject"], match["audio"])].append(match["cond"])

# Match runs to standardised trials per (subject, stimulus). Runs that have EEG
# claim them first, in run order; the count is what is reliable here, so where a
# subject has several runs of one stimulus the pairing is by order, not identity.
trials["condition"] = pd.NA
trials["standardised"] = False
for key, group in trials.groupby(["subject", "stimulus"], sort=False):
    available = list(standardised.get(key, []))
    for index in group.sort_values(["has_eeg", "run"], ascending=[False, True]).index:
        if not available or not trials.at[index, "has_eeg"]:
            continue
        trials.at[index, "condition"] = available.pop(0)
        trials.at[index, "standardised"] = True

In [ ]:
def exclusion_reason(row) -> str:
    """Why this run is not in the analysis; empty string if it is."""
    if not row["has_eeg"]:
        return NOT_RECORDED
    if not row["standardised"]:
        return NOT_STANDARDISED

    reasons = []
    if row["condition"] in CONDITION_REASONS:
        reasons.append(CONDITION_REASONS[row["condition"]])
    reasons += [r for marker, r in RECORDING_REASONS.items() if marker in row["stimulus"]]
    return "; ".join(reasons)


trials["exclusion_reason"] = trials.apply(exclusion_reason, axis=1)
trials["excluded"] = trials["exclusion_reason"] != ""

trials = trials[["subject", "stimulus", "session", "run", "condition",
                 "hours", "has_eeg", "standardised", "excluded", "exclusion_reason"]]

analysed = trials[~trials["excluded"]]
print(f"{len(trials)} listening runs presented ({resting_runs} resting-state runs ignored)")
print(f"  {int(trials['has_eeg'].sum())} with an EEG file, "
      f"{int(trials['standardised'].sum())} standardised, {len(analysed)} analysed")
print(f"  {analysed.stimulus.nunique()} stimuli, {analysed.subject.nunique()} subjects, "
      f"{analysed.hours.sum():.1f} hours")
lost_subjects = sorted(set(trials.subject) - set(analysed.subject))
print(f"  {trials.subject.nunique()} subjects presented, "
      f"{len(lost_subjects)} contribute nothing: {', '.join('sub-' + s for s in lost_subjects)}")
assert analysed.stimulus.nunique() == len(STIMULI), "audit disagrees with the inventory above"

trials.head(12)

In [ ]:
# Excluded trials by cause
by_reason = (
    trials[trials["excluded"]]
    .groupby("exclusion_reason")
    .agg(trials=("run", "size"), subjects=("subject", "nunique"),
         stimuli=("stimulus", "nunique"), hours=("hours", "sum"))
    .sort_values("trials", ascending=False)
    .round({"hours": 2})
)
by_reason

In [ ]:
# Excluded trials by stimulus — every stimulus losing at least one trial.
# Each column is aggregated over the subset it names, not over all rows.
dropped = trials[trials["excluded"]]
by_stimulus = pd.DataFrame({
    "trials_total": trials.groupby("stimulus").size(),
    "trials_analysed": analysed.groupby("stimulus").size(),
    "trials_excluded": dropped.groupby("stimulus").size(),
    "subjects_analysed": analysed.groupby("stimulus")["subject"].nunique(),
    "hours_excluded": dropped.groupby("stimulus")["hours"].sum(),
    "reasons": dropped.groupby("stimulus")["exclusion_reason"].agg(
        lambda col: "; ".join(sorted({r for r in col if r}))
    ),
})
counts = ["trials_total", "trials_analysed", "trials_excluded", "subjects_analysed"]
by_stimulus[counts] = by_stimulus[counts].fillna(0).astype(int)
by_stimulus["hours_excluded"] = by_stimulus["hours_excluded"].fillna(0.0).round(2)
by_stimulus["reasons"] = by_stimulus["reasons"].fillna("")

by_stimulus = by_stimulus[by_stimulus["trials_excluded"] > 0].sort_values(
    "trials_excluded", ascending=False
)
by_stimulus

### Subjects

Four subjects never reach the analysis at all: every run they were presented is in
`events.tsv` but has no EEG file in the download. They are in `trials` like anyone
else — they simply have no row with `excluded == False`.

In [ ]:
by_subject = pd.DataFrame({
    "session": trials.groupby("subject")["session"].first(),
    "runs_presented": trials.groupby("subject").size(),
    "runs_with_eeg": trials.groupby("subject")["has_eeg"].sum(),
    "trials_analysed": analysed.groupby("subject").size(),
    "hours_analysed": analysed.groupby("subject")["hours"].sum(),
})
subject_counts = ["runs_presented", "runs_with_eeg", "trials_analysed"]
by_subject[subject_counts] = by_subject[subject_counts].fillna(0).astype(int)
by_subject["hours_analysed"] = by_subject["hours_analysed"].fillna(0.0).round(2)

incomplete = by_subject[by_subject["runs_with_eeg"] < by_subject["runs_presented"]]
print(f"{len(by_subject)} subjects presented; "
      f"{int((by_subject['trials_analysed'] == 0).sum())} contribute no analysed trial")
incomplete

### Did every subject hear audiobook 1?

Audiobook 1 is the stimulus most subjects share, but it was presented in several
forms: whole (`audiobook_1`), split into halves (`audiobook_1_1`, `audiobook_1_2`),
time-shifted halves, and one artefact take. This asks, per subject, whether *any*
of them was presented, whether it was recorded, and whether it survives into the
analysis.

In [ ]:
# Anything named audiobook_1 or audiobook_1_*, but not audiobook_10..audiobook_14.
is_family = trials["stimulus"].str.match(r"audiobook_1(_|$)")
family = trials[is_family]

all_subjects = sorted(trials["subject"].unique())
audiobook_1_coverage = pd.DataFrame({
    "presented": family.groupby("subject")["stimulus"].agg(lambda col: ", ".join(sorted(col))),
    "runs": family.groupby("subject").size(),
    "runs_with_eeg": family.groupby("subject")["has_eeg"].sum(),
    "trials_analysed": family[~family["excluded"]].groupby("subject").size(),
}).reindex(all_subjects)
audiobook_1_coverage["presented"] = audiobook_1_coverage["presented"].fillna("")
coverage_counts = ["runs", "runs_with_eeg", "trials_analysed"]
audiobook_1_coverage[coverage_counts] = audiobook_1_coverage[coverage_counts].fillna(0).astype(int)
audiobook_1_coverage.index.name = "subject"

never_presented = audiobook_1_coverage.index[audiobook_1_coverage["runs"] == 0]
no_recording = audiobook_1_coverage.index[
    (audiobook_1_coverage["runs"] > 0) & (audiobook_1_coverage["runs_with_eeg"] == 0)
]
not_analysed = audiobook_1_coverage.index[audiobook_1_coverage["trials_analysed"] == 0]

print(f"{len(all_subjects)} subjects total")
print(f"  presented some audiobook-1 variant: {len(all_subjects) - len(never_presented)}")
print(f"  never presented any:                {len(never_presented)}"
      f"  {', '.join('sub-' + s for s in never_presented)}")
print(f"  presented but nothing recorded:     {len(no_recording)}"
      f"  {', '.join('sub-' + s for s in no_recording)}")
print(f"  with an analysed audiobook-1 trial: {len(all_subjects) - len(not_analysed)}")

# Subjects that did not get the whole audiobook_1, only parts or shifted variants
partial = audiobook_1_coverage[
    (audiobook_1_coverage["runs"] > 0)
    & ~audiobook_1_coverage["presented"].str.split(", ").apply(lambda s: "audiobook_1" in s)
]
print(f"\n{len(partial)} subject(s) got only parts/variants, never the whole recording:")
partial

### Exclusion table for the paper

The same exclusions as above, rebuilt from the two logs written when the data were fetched and
standardised, and printed as a LaTeX `longtable` (needs `\usepackage{longtable}`):

- **`download_log.tsv`.** Raw EEG the repository refused with HTTP 403, so it is not publicly available.
  Also runs whose `events.tsv` is published but which have no EEG file listed at all.
- **`standardise_log.tsv`.** Trials whose EEG triggers could not be aligned to the stimulus, plus the
  condition and stimulus name of every trial that did standardise.

The standardise log is appended to on every run, and it currently holds two complete passes, so
each trial's **last** entry is the one used. "unclosed file" warnings are dropped first. They are
resource warnings from the previous trial's stimulus file, not something wrong with the trial
they are logged against.

In [ ]:
DATASET_STORAGE = Path("/data_storage/gu08wuvu/datasets/accou2023")
DOWNLOAD_LOG = DATASET_STORAGE / "downloaded" / "download_log.tsv"
STANDARDISE_LOG = DATASET_STORAGE / "standardised-flat" / "standardise_log.tsv"

NOT_AVAILABLE = "EEG data not publicly available"
NOT_PUBLISHED = "No EEG recording in the published dataset"
TRIGGER_ERROR = "EEG triggers could not be aligned to the stimulus"
# The table's wording for each cause, and the order the groups appear in.
TABLE_CONDITION_REASONS = {
    "sin": "Speech presented in background noise",
    "av": "Audiovisual presentation",
}
TABLE_STIMULUS_REASONS = {
    "_shifted": "Pitch-shifted stimulus",
    "_artefact": "Stimulus not delivered to participant",
}
REASON_ORDER = [NOT_AVAILABLE, NOT_PUBLISHED, TRIGGER_ERROR, *TABLE_CONDITION_REASONS.values(),
                *TABLE_STIMULUS_REASONS.values()]

DOWNLOAD_EVENTS_PATTERN = re.compile(
    r"sub-(?P<subject>\d+)/ses-[^/]+/eeg/sub-\d+_ses-[^_]+_task-listeningActive"
    r"_run-(?P<run>\d+)_events\.tsv$")
DOWNLOAD_EEG_PATTERN = re.compile(
    r"sub-(?P<subject>\d+)/ses-[^/]+/eeg/sub-\d+_ses-[^_]+_task-listeningActive"
    r"_run-(?P<run>\d+)_eeg\.bdf\.gz$")


def run_ranges(runs):
    """[1, 2, 3, 5] -> '1--3, 5'."""
    runs, spans = sorted(runs), []
    start = previous = runs[0]
    for run in runs[1:] + [None]:
        if run is not None and run == previous + 1:
            previous = run
            continue
        spans.append(f"{start}" if start == previous else f"{start}--{previous}")
        if run is not None:
            start = previous = run
    return ", ".join(spans)


# --- raw EEG the repository would not serve ------------------------------------
download_log = pd.read_csv(DOWNLOAD_LOG, sep="\t", dtype=str)
raw_eeg = download_log["file"].str.extract(DOWNLOAD_EEG_PATTERN).dropna()
raw_eeg["run"] = raw_eeg["run"].astype(int)
raw_eeg["status"] = download_log.loc[raw_eeg.index, "status"]

table_rows = []
for subject, runs in raw_eeg.groupby("subject"):
    refused = runs[runs["status"] == "failed"]
    if refused.empty:
        continue
    table_rows.append({
        "subject": subject,
        "run": "all" if len(refused) == len(runs) else run_ranges(refused["run"].tolist()),
        "stimulus": "-",
        "reason": NOT_AVAILABLE,
        "sort_run": 0,
    })

# --- runs that were presented but whose EEG was never published --------------
# The run has an events.tsv in the repository, but no EEG file is listed for it at all: not
# refused like the rows above, just absent. Runs whose events.tsv names no stimulus are skipped,
# as in the audit above.
listed_files = set(download_log["file"])
for file in download_log["file"]:
    match = DOWNLOAD_EVENTS_PATTERN.match(file)
    if match is None or file.replace("_events.tsv", "_eeg.bdf.gz") in listed_files:
        continue
    with (DATASET_STORAGE / "downloaded" / file).open(newline="") as handle:
        event = next(iter(csv.DictReader(handle, delimiter="\t")), None)
    stim_file = (event or {}).get("stim_file", "")
    if not stim_file or stim_file == "n/a":
        continue
    table_rows.append({
        "subject": match["subject"],
        "run": match["run"].lstrip("0") or "0",
        "stimulus": Path(stim_file).name.replace(".npz.gz", ""),
        "reason": NOT_PUBLISHED,
        "sort_run": int(match["run"]),
    })

# --- trials that standardised or failed to ------------------------------------
standardise_log = pd.read_csv(STANDARDISE_LOG, sep="\t", dtype=str, keep_default_na=False)
standardise_log = standardise_log[
    (standardise_log["subject"] != "")
    & ~standardise_log["message"].str.startswith("unclosed file")
]
latest = standardise_log.groupby(["subject", "session", "trial"], sort=False).tail(1)

for row in latest.itertuples(index=False):
    if row.status == "failed":
        reason = TRIGGER_ERROR
    elif row.condition in TABLE_CONDITION_REASONS:
        reason = TABLE_CONDITION_REASONS[row.condition]
    else:
        reason = next((text for marker, text in TABLE_STIMULUS_REASONS.items()
                       if marker in row.stimulus), None)
    if reason is not None:
        table_rows.append({
            "subject": f"{int(row.subject):03d}",
            "run": row.trial,
            "stimulus": row.stimulus,
            "reason": reason,
            "sort_run": int(row.trial),
        })

exclusion_table = pd.DataFrame(table_rows)
exclusion_table["group"] = exclusion_table["reason"].map(REASON_ORDER.index)
exclusion_table = (exclusion_table.sort_values(["group", "subject", "sort_run"])
                   .drop(columns=["group", "sort_run"]).reset_index(drop=True))

# Cross-check against the audit above, which counts from events.tsv and the files on disk.
# Every standardised trial excluded there has a row here, and vice versa.
logged_trial_rows = exclusion_table[~exclusion_table["reason"].isin([NOT_AVAILABLE,
                                                                      NOT_PUBLISHED])]
audit_standardised_excluded = int((trials["standardised"] & trials["excluded"]).sum())
audit_not_standardised = int((trials["has_eeg"] & ~trials["standardised"]).sum())
assert len(logged_trial_rows) == audit_standardised_excluded + audit_not_standardised, (
    f"{len(logged_trial_rows)} logged exclusions vs {audit_standardised_excluded} + "
    f"{audit_not_standardised} in the audit")
# Every run without EEG in the audit is either refused (403) or never published.
refused_runs = raw_eeg[raw_eeg["status"] == "failed"][["subject", "run"]]
audit_no_eeg = trials[~trials["has_eeg"]].assign(run=lambda frame: frame["run"].astype(int))
refused_with_stimulus = audit_no_eeg.merge(refused_runs, on=["subject", "run"])
unpublished_count = int((exclusion_table["reason"] == NOT_PUBLISHED).sum())
assert len(refused_with_stimulus) + unpublished_count == len(audit_no_eeg), (
    f"{len(refused_with_stimulus)} refused + {unpublished_count} unpublished vs "
    f"{len(audit_no_eeg)} runs without EEG in the audit")


def latex_escape(text):
    return str(text).replace("_", r"\_").replace("&", r"\&").replace("%", r"\%")


# A longtable, which needs \usepackage{longtable} in the preamble. The header row is repeated on
# every page and every page but the last says the table continues.
HEADER = r" Participant & Run & Stimulus & Reason \\ [0.5ex]"
lines = [
    r"\begin{longtable}{||c c c c||}",
    r" \hline",
    HEADER,
    r" \hline\hline",
    r" \endfirsthead",
    r" \hline",
    HEADER,
    r" \hline\hline",
    r" \endhead",
    r" \multicolumn{4}{r}{\textit{Continued on next page}} \\",
    r" \endfoot",
    r" \endlastfoot",
]
for index, row in enumerate(exclusion_table.itertuples(index=False)):
    last = index == len(exclusion_table) - 1
    cells = " & ".join(latex_escape(value) for value in
                       (row.subject, row.run, row.stimulus, row.reason))
    lines.append(f" {cells} \\\\" + (" [1ex]" if last else ""))
    lines.append(r" \hline")
lines.append(r"\end{longtable}")
EXCLUSION_LATEX = "\n".join(lines)

print(exclusion_table.groupby("reason", sort=False).size().to_string(), "\n")
print(EXCLUSION_LATEX)

### How many subjects and which trials did not have external mastoids?

The linked-mastoid reference needs both `A1` and `A2`. They are `misc`-typed in
these recordings, so they sit outside `pick("eeg")` and never enter the data — but
a recording that carries neither has no linked-mastoid reference to apply at all,
and the pipeline skips it, so its duration never reaches that reference's
denominator. The average and FCz references are unaffected.

The scan below reads FIF headers only, no sample data, which is the same check
`reference_channel_names` makes in `src/coherence/pipeline.py`. It is slow enough
over NFS to be worth caching, so the result is written to `figures/figure_data/`
and reused on later runs.

In [ ]:
MASTOID_CHANNELS = ("A1", "A2")
MASTOID_CACHE = ROOT / "figures" / "figure_data" / "mastoid_channels.csv"
HEADER_SCAN_WORKERS = 12


def mastoids_present(fif_path):
    """Which of A1/A2 this recording carries, from its header alone."""
    info = mne.io.read_info(fif_path, verbose="ERROR")
    return [name for name in MASTOID_CHANNELS if name in info["ch_names"]]


if MASTOID_CACHE.exists():
    mastoids = pd.read_csv(MASTOID_CACHE, dtype={"subject": str})
    mastoids["mastoids"] = mastoids["mastoids"].fillna("")
else:
    # ~2 s per header over NFS, so 617 of them in series is ~25 minutes. Processes,
    # not threads: the wait is NFS latency, but FIF header parsing holds the GIL.
    paths = sorted(EEG_DIR.glob("*.fif"))
    with ProcessPoolExecutor(max_workers=HEADER_SCAN_WORKERS) as pool:
        found = list(tqdm(pool.map(mastoids_present, paths, chunksize=4),
                          total=len(paths), desc="FIF headers", unit="file"))

    rows = []
    for path, names in zip(paths, found):
        match = FLAT_PATTERN.match(path.name)
        if match is None:
            continue
        rows.append({
            "subject": match["subject"],
            "stimulus": match["audio"],
            "condition": match["cond"],
            "mastoids": ", ".join(names),
            "n_mastoids": len(names),
        })
    mastoids = pd.DataFrame(rows)
    MASTOID_CACHE.parent.mkdir(parents=True, exist_ok=True)
    mastoids.to_csv(MASTOID_CACHE, index=False)

# The inventory's exclusions again, so "analysed" here means the same trials it counted.
mastoids["analysed"] = ~(
    mastoids["condition"].isin(EXCLUDED_CONDITIONS)
    | mastoids["stimulus"].str.contains("|".join(EXCLUDED_MARKERS), regex=True)
)
mastoids["has_both"] = mastoids["n_mastoids"] == len(MASTOID_CHANNELS)

without = mastoids[~mastoids["has_both"]]
analysed_without = without[without["analysed"]]
partial = without[without["n_mastoids"] == 1]

print(f"{len(mastoids)} standardised recordings, "
      f"{int(mastoids['has_both'].sum())} carrying both A1 and A2")
print(f"  {len(without)} carry neither ({len(partial)} carry exactly one)")
print(f"  {len(analysed_without)} of the {int(mastoids['analysed'].sum())} analysed trials "
      f"are affected, from {analysed_without.subject.nunique()} of "
      f"{mastoids[mastoids['analysed']].subject.nunique()} subjects: "
      f"{', '.join('sub-' + s for s in sorted(analysed_without.subject.unique()))}")

In [ ]:
# Per subject: mastoids are a property of the capping, so they are all-or-nothing
# within a subject unless a session was re-capped. Only subjects losing a trial appear.
by_subject_mastoids = pd.DataFrame({
    "recordings": mastoids.groupby("subject").size(),
    "with_both": mastoids.groupby("subject")["has_both"].sum(),
    "analysed": mastoids[mastoids["analysed"]].groupby("subject").size(),
    "analysed_with_both": mastoids[mastoids["analysed"]].groupby("subject")["has_both"].sum(),
    "stimuli_lost": without[without["analysed"]].groupby("subject")["stimulus"].agg(
        lambda col: ", ".join(sorted(col))
    ),
})
counts = ["recordings", "with_both", "analysed", "analysed_with_both"]
by_subject_mastoids[counts] = by_subject_mastoids[counts].fillna(0).astype(int)
by_subject_mastoids["stimuli_lost"] = by_subject_mastoids["stimuli_lost"].fillna("")

by_subject_mastoids[by_subject_mastoids["with_both"] < by_subject_mastoids["recordings"]]

In [ ]:
# Every standardised recording without both mastoids, one row per trial.
without[["subject", "stimulus", "condition", "mastoids", "analysed"]].sort_values(
    ["subject", "stimulus"]
).reset_index(drop=True)

## Supplementary details

A two-panel figure describing the stimulus set from two angles the main figure
does not cover.

1. **Panel a** -- the envelope modulation spectrum in its area-preserving form,
   `f * P(f)` on a log frequency axis over 0.2-32 Hz, with and without the longer
   silences. `SUPP_SPECTRA_UNITS` picks the scale: "physical" (the default) draws it in
   Pa$^{1.2}$, from each stimulus's calibrated 62 dB(A) presentation level, so the two
   curves keep their real levels; "band_normalised" scales each to unit area over the band,
   comparing shapes only. On a log axis `f * P(f)` is what preserves area, so the syllabic peak
   becomes a peak rather than a shoulder on the 1/f background (see
   `analysis/modulation_spectra.ipynb`). Pauses between utterances feed the
   sub-1 Hz end, so removing them is what separates prosodic structure from
   silence; the caveat is that the retained segments are spliced, and the
   crossfaded joins shape the low end too.
2. **Panel b** -- Pearson correlations among four unit rates, one per level of the
   linguistic hierarchy, over the 69 stimuli. The rates are not independent
   measurements of different things, and this is how much of the set is a single
   shared "speaking rate" axis. Kept deliberately narrow: the phonemic categories
   of panel d are the phoneme rate's own parts, and only the de Jong syllable
   estimator is carried, since panel c is where the two estimators are compared.

In [ ]:
# Silence detection, as in `analysis/modulation_spectra.ipynb`. The envelope noise
# floor sits at a different level in every recording, so the speech/silence cut is
# placed per stimulus, at the valley of the level distribution rather than at a
# fixed threshold.

SUPP_BAND = (0.2, 32.0)


def otsu_threshold(values, lo=-60.0, hi=5.0, n_bins=256):
    """Cut a bimodal distribution at the valley between its two modes."""
    counts, _ = np.histogram(np.clip(values, lo, hi), bins=n_bins, range=(lo, hi))
    edges = np.linspace(lo, hi, n_bins + 1)
    centres = 0.5 * (edges[1:] + edges[:-1])

    p = counts / counts.sum()
    w0 = np.cumsum(p)
    w1 = 1.0 - w0
    cumulative_mean = np.cumsum(p * centres)
    total_mean = cumulative_mean[-1]

    valid = (w0 > 1e-6) & (w1 > 1e-6)
    variance = np.zeros(n_bins)
    variance[valid] = (
        (total_mean * w0[valid] - cumulative_mean[valid]) ** 2 / (w0[valid] * w1[valid])
    )
    return float(centres[np.argmax(variance)])


def _runs(mask, value):
    """Start/stop sample indices of each run of `value` in a boolean mask."""
    flags = (mask == value).view(np.int8)
    edges = np.flatnonzero(np.diff(np.concatenate(([0], flags, [0]))))
    return edges.reshape(-1, 2)


def find_speech_segments(envelope, fs=ENVELOPE_FS, smooth_ms=50.0, min_silence_ms=150.0,
                         min_speech_ms=100.0, pad_ms=25.0):
    """Sample ranges of the envelope that are speech rather than silence.

    Silences shorter than `min_silence_ms` are kept, since they are speech-internal
    (stop closures); speech runs shorter than `min_speech_ms` are dropped; and what
    is retained is padded by `pad_ms` so onsets and offsets survive.
    """
    envelope = np.asarray(envelope, dtype=float).squeeze()

    window = max(1, int(round(smooth_ms * 1e-3 * fs)))
    smoothed = np.convolve(np.abs(envelope), np.ones(window) / window, mode="same")
    reference = max(np.percentile(smoothed, 95), 1e-12)
    level_db = 20 * np.log10(np.maximum(smoothed, 1e-12) / reference)
    is_speech = level_db > otsu_threshold(level_db)

    for start, stop in _runs(is_speech, False):
        if stop - start < int(round(min_silence_ms * 1e-3 * fs)):
            is_speech[start:stop] = True
    for start, stop in _runs(is_speech, True):
        if stop - start < int(round(min_speech_ms * 1e-3 * fs)):
            is_speech[start:stop] = False

    pad = int(round(pad_ms * 1e-3 * fs))
    segments = []
    for start, stop in _runs(is_speech, True):
        start, stop = max(0, start - pad), min(envelope.size, stop + pad)
        if segments and start <= segments[-1][1]:
            segments[-1] = (segments[-1][0], max(segments[-1][1], stop))
        else:
            segments.append((start, stop))
    return segments


def concatenate_segments(envelope, segments, fs=ENVELOPE_FS, crossfade_ms=20.0):
    """Splice segments together, crossfading the joins to limit spectral splatter."""
    envelope = np.asarray(envelope, dtype=float).squeeze()
    crossfade = int(round(crossfade_ms * 1e-3 * fs))

    pieces = [envelope[start:stop] for start, stop in segments if stop > start]
    if not pieces:
        return np.zeros(0)

    out = pieces[0].copy()
    for piece in pieces[1:]:
        n = min(crossfade, out.size, piece.size)
        if n < 1:
            out = np.concatenate([out, piece])
            continue
        ramp = np.linspace(0.0, 1.0, n, endpoint=False)
        out[-n:] = out[-n:] * (1.0 - ramp) + piece[:n] * ramp
        out = np.concatenate([out, piece[n:]])
    return out


def envelope_psd(signal):
    """The same Welch estimate the modulation-spectra panels use."""
    return welch(signal, fs=ENVELOPE_FS, nperseg=int(ENVELOPE_FS * 5),
                 noverlap=int(ENVELOPE_FS * 4))

In [ ]:
# Both spectra per stimulus, in STIMULI order so the pair stays matched. The raw
# ones repeat panel e's estimate; they are recomputed rather than reused because
# `psds` there is sorted by median F0, not by stimulus.
raw_psds, trimmed_psds, removed_fraction = [], [], []
for name in STIMULI:
    with np.load(FEATURES_DIR / f"{name}.npz") as features:
        envelope = np.clip(features["gammatone_envelope"].squeeze(), 0, None)

    supp_frequencies, power = envelope_psd(envelope)
    raw_psds.append(power)

    trimmed = concatenate_segments(envelope, find_speech_segments(envelope))
    removed_fraction.append(1 - trimmed.size / envelope.size)
    trimmed_psds.append(envelope_psd(trimmed)[1])

raw_psds = np.array(raw_psds)
trimmed_psds = np.array(trimmed_psds)
removed_fraction = np.array(removed_fraction)

in_supp_band = (supp_frequencies >= SUPP_BAND[0]) & (supp_frequencies <= SUPP_BAND[1])


def area_normalised(psd):
    """`f * P(f)`, scaled to unit area under the log-frequency axis of the panel.

    On a log axis `f * P(f)` is the area-preserving form, so normalising by that
    same area is what makes two spectra at different absolute levels -- removing
    silence raises the level -- comparable in shape.
    """
    weighted = supp_frequencies * np.asarray(psd)
    area = np.trapezoid(weighted[..., in_supp_band],
                        np.log10(supp_frequencies[in_supp_band]), axis=-1)
    return weighted / np.asarray(area)[..., None]


# How panel a draws the spectra. "physical": each stimulus's PSD converted to Pa^1.2/Hz from its
# calibrated presentation level (62 dB(A); the factors notebook 6 computes and caches), and drawn
# as f * P(f) in Pa^1.2 with no normalisation, so the two curves keep their real levels.
# "band_normalised": f * P(f) scaled to unit area over SUPP_BAND, which compares shapes only.
SUPP_SPECTRA_UNITS = "physical"            # "physical" | "band_normalised"

if SUPP_SPECTRA_UNITS == "physical":
    # The envelope is compressed (power 0.6), so its PSD scales as k^1.2 for a waveform gain k.
    calibration_cache = ROOT / "figures" / "figure_data" / "stimulus_pressure_calibration.npz"
    if not calibration_cache.exists():
        raise FileNotFoundError(f"{calibration_cache.name} is written by notebook 6; run it first")
    with np.load(calibration_cache) as saved:
        rms_a = dict(zip([str(n) for n in saved["stimuli"]], saved["rms_a_tube"]))
    pressure_62 = 20e-6 * 10 ** (62.0 / 20)
    POWER_FACTOR = np.array([(pressure_62 / rms_a[name]) ** 1.2 for name in STIMULI])[:, None]
    scale = lambda psd: supp_frequencies * np.asarray(psd)
    raw_weighted = scale(raw_psds * POWER_FACTOR)
    trimmed_weighted = scale(trimmed_psds * POWER_FACTOR)
    raw_weighted_average = scale(np.average(raw_psds * POWER_FACTOR, axis=0, weights=WEIGHTS))
    trimmed_weighted_average = scale(np.average(trimmed_psds * POWER_FACTOR, axis=0,
                                                weights=WEIGHTS))
    SUPP_YLABEL = r"$f\,P(f)$ (Pa$^{1.2}$)"
else:
    raw_weighted = area_normalised(raw_psds)
    trimmed_weighted = area_normalised(trimmed_psds)
    raw_weighted_average = area_normalised(np.average(raw_psds, axis=0, weights=WEIGHTS))
    trimmed_weighted_average = area_normalised(np.average(trimmed_psds, axis=0, weights=WEIGHTS))
    SUPP_YLABEL = r"$f\,P(f)$ (band-normalised)"

SUPP_PEAK_BAND = (supp_frequencies >= 1) & (supp_frequencies <= 10)
SUPP_PEAKS = {
    label: float(supp_frequencies[SUPP_PEAK_BAND][np.argmax(average[SUPP_PEAK_BAND])])
    for label, average in [("whole stimulus", raw_weighted_average),
                           ("silences removed", trimmed_weighted_average)]
}

print(f"silence removed: {100 * removed_fraction.min():.0f}-{100 * removed_fraction.max():.0f}% "
      f"of each stimulus (median {100 * np.median(removed_fraction):.0f}%)")
print("f*P(f) peak: " + ", ".join(f"{label} {peak:.1f} Hz" for label, peak in SUPP_PEAKS.items()))

In [ ]:
# The four unit rates that span the linguistic hierarchy, one row per stimulus.
# Deliberately narrow: the phonemic categories are the phoneme rate's own parts and
# correlate with it at 0.98-1.00, and only one syllable estimator is carried, since
# panel c of the main figure is already the place the two are compared.
STIMULUS_MEASURES = {
    "Intonation units": iu_rates,
    "Words": word_rates,
    "Syllables (de Jong)": syllable_rates_dejong,
    "Phonemes": phoneme_rates,
}

measures = pd.DataFrame(STIMULUS_MEASURES).reindex(STIMULI)
assert not measures.isna().to_numpy().any(), "a measure is missing for some stimulus"

# Pearson: the question is how much of one rate is linear in another, and these are
# rates on a common scale rather than ranks. scipy's pearsonr takes a pair at a
# time, so the matrix is filled pair by pair -- six of them here.
MEASURE_LABELS = list(measures.columns)
columns = measures.to_numpy().T

CORRELATIONS = np.eye(len(MEASURE_LABELS))
CORRELATION_P = np.zeros_like(CORRELATIONS)
for i in range(len(MEASURE_LABELS)):
    for j in range(i):
        r, p = pearsonr(columns[i], columns[j])
        CORRELATIONS[i, j] = CORRELATIONS[j, i] = r
        CORRELATION_P[i, j] = CORRELATION_P[j, i] = p

# Benjamini-Hochberg over the unique pairs, not over the full square.
LOWER_TRIANGLE = np.tril_indices(len(MEASURE_LABELS), -1)
SIGNIFICANT = np.zeros_like(CORRELATIONS, dtype=bool)
SIGNIFICANT[LOWER_TRIANGLE] = false_discovery_control(CORRELATION_P[LOWER_TRIANGLE]) < 0.05

print(f"{measures.shape[0]} stimuli x {measures.shape[1]} measures; "
      f"{int(SIGNIFICANT.sum())} of {len(LOWER_TRIANGLE[0])} pairs significant (FDR < 0.05)")
print(f"|r| median {np.median(np.abs(CORRELATIONS[LOWER_TRIANGLE])):.2f}, "
      f"range {np.min(np.abs(CORRELATIONS[LOWER_TRIANGLE])):.2f}"
      f"-{np.max(np.abs(CORRELATIONS[LOWER_TRIANGLE])):.2f}")

In [ ]:
# Two lines rather than a rotation: the names sit under their own columns at this
# size once the long ones are broken, and horizontal labels are easier to read.
WRAPPED_LABELS = {
    "Intonation units": "Intonation\nunits",
    "Syllables (de Jong)": "Syllables\n(de Jong)",
}


def wrapped(label):
    return WRAPPED_LABELS.get(label, label)


CORRELATION_CMAP = plt.get_cmap("RdBu_r")
CORRELATION_NORM = mcolors.Normalize(vmin=-1, vmax=1)


SPECTRUM_COLOURS = {"whole stimulus": INK, "silences removed": BAR}

# SYLLABLE_COLOUR is figure 2's syllable colour, defined with the palette in the setup
# cell: main panel g's syllabic-rate box uses it too, so the quantity is the same
# colour wherever it is marked as a series of its own.


def plot_weighted_spectra(ax):
    """`f * P(f)` on a log axis, whole stimulus against silences removed.

    Drawn the way main panel g is: every stimulus a light trace of its own, with
    the listening-time-weighted mean over them at full strength on top.
    """
    for values, average, label in [
        (raw_weighted, raw_weighted_average, "whole stimulus"),
        (trimmed_weighted, trimmed_weighted_average, "silences removed"),
    ]:
        colour = SPECTRUM_COLOURS[label]
        # zorder 1: below the histogram. Left to the default it is 2, which put the
        # 138 faint traces over the histogram and buried it.
        ax.plot(supp_frequencies, values.T, color=colour, lw=0.4, alpha=0.16, zorder=1)
        ax.plot(supp_frequencies, average, color=colour, lw=1.8, zorder=3)
        ax.plot([SUPP_PEAKS[label]], [average[SUPP_PEAK_BAND].max()], marker="v", ms=4,
                color=colour, clip_on=False, zorder=3)

    # Headroom above the traces. It used to hold a box of the across-stimulus syllable
    # rates; that is gone -- the histogram below carries the same variable and more of
    # it -- and the band now keeps the legend clear of the traces instead.
    top = max(raw_weighted.max(), trimmed_weighted.max()) * 1.10

    # Figure 2's syllabic-rate histogram, on the spectra's own frequency axis. This is
    # the WITHIN-stimulus spread: how fast the speech runs from one 5 s window to the
    # next, pooled over the set by listening time. Left off the y axis, because it is
    # context for where the spectra's syllabic peak sits rather than a third curve to
    # read a value off -- its own units are counts.
    #
    # The stack of this panel, low to high: individual traces (1), histogram fill (2)
    # and its outline (2.5), the two weighted averages and their peak markers (3), the
    # legend (5). The histogram clears the faint traces so its shape is readable, and
    # stays under the averages so it never competes with the result.
    HISTOGRAM_PEAK = 0.15         # of the panel height, so it stays under the traces
    visible = rate_centres > 0      # the zero bin has no place on a log axis
    histogram = pooled_dejong[visible] / pooled_dejong.max() * top * HISTOGRAM_PEAK
    ax.fill_between(rate_centres[visible], histogram, step="mid",
                    color=SYLLABLE_COLOUR, alpha=0.75, lw=0, zorder=2)
    ax.step(rate_centres[visible], histogram, where="mid", color=SYLLABLE_COLOUR,
            lw=0.9, alpha=1, zorder=2.5)

    ax.set_ylim(0, top)

    ax.set_xscale("log")
    ax.set_xlim(*SUPP_BAND)
    # Octaves, and majors only: a log axis puts minor ticks between every decade
    # by default, which at this size reads as noise along the spine.
    ax.set_xticks([0.25, 0.5, 1, 2, 4, 8, 16, 32],
                  ["0.25", "0.5", "1", "2", "4", "8", "16", "32"], fontsize=TICK_FONTSIZE)
    ax.tick_params(axis="x", which="minor", length=0)
    ax.set_xlabel("Modulation frequency (Hz)")
    ax.set_ylabel(SUPP_YLABEL)
    ax.set_title("Envelope modulation spectra", fontsize=DEFAULT_FONTSIZE)
    # Swatches at one weight: the plotted lines keep their own widths, but in the
    # legend the difference between them is identity, not magnitude.
    ax.legend(
        [Line2D([], [], color=SPECTRUM_COLOURS["whole stimulus"], lw=1.5),
         Line2D([], [], color=SPECTRUM_COLOURS["silences removed"], lw=1.5),
         Rectangle((0, 0), 1, 1, facecolor=SYLLABLE_COLOUR, alpha=0.75,
                   edgecolor="none")],
        ["With pauses", "Pauses removed", "Syllabic rate (5 s windows)"],
        # Upper left, into the headroom the box used to occupy. Lower left sat on the
        # histogram's shoulder; the right-hand corners are where the traces peak. The
        # left of the band is the one corner nothing reaches -- the traces top out at
        # 3-5 Hz, well inside the panel.
        loc="upper left", fontsize=TICK_FONTSIZE,
        handlelength=1.5, labelspacing=0.3, borderpad=0.4,
        # A white wash rather than a rule: enough to lift the labels off anything
        # behind them without drawing a box the eye has to read past.
        frameon=True, fancybox=False, framealpha=0.6,
        facecolor="white", edgecolor="white",
    ).set_zorder(5)
    return bare(ax)


def plot_measure_correlations(ax):
    """Lower triangle of the Spearman matrix; bold entries survive FDR < 0.05."""
    n = len(MEASURE_LABELS)
    shown = np.full((n, n), np.nan)
    shown[LOWER_TRIANGLE] = CORRELATIONS[LOWER_TRIANGLE]
    ax.imshow(np.ma.masked_invalid(shown), cmap=CORRELATION_CMAP, norm=CORRELATION_NORM)

    for i, j in zip(*LOWER_TRIANGLE):
        rho = CORRELATIONS[i, j]
        ax.text(j, i, f"{rho:.2f}".replace("0.", ".").replace("-.", "\u2212."),
                ha="center", va="center", fontsize=7,
                fontweight="bold" if SIGNIFICANT[i, j] else "normal",
                color=INK if abs(rho) < 0.6 else "white")

    ax.set_xticks(range(n - 1))
    ax.set_xticklabels([wrapped(label) for label in MEASURE_LABELS[:-1]],
                       rotation=0, ha="center", fontsize=SMALL_FONTSIZE)
    ax.set_yticks(range(1, n))
    ax.set_yticklabels([wrapped(label) for label in MEASURE_LABELS[1:]],
                       fontsize=SMALL_FONTSIZE)
    ax.set_xlim(-0.5, n - 1.5)
    ax.set_ylim(n - 0.5, 0.5)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title("Unit rate correlations ($r$)", fontsize=DEFAULT_FONTSIZE, pad=10)
    return ax

In [ ]:
# Side by side, and squarer than the text width would make it: the numbers in b
# carry the scale on their own, so neither panel needs the width a colourbar did.
fig = plt.figure(figsize=(5.6, 3.0), layout="constrained")
gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1, 1])

ax_spectra = fig.add_subplot(gs[0])
ax_correlations = fig.add_subplot(gs[1])

plot_weighted_spectra(ax_spectra)
plot_measure_correlations(ax_correlations)

# The main figure's helper, so both figures letter their panels the same way and
# the two letters share a height rather than each sitting over its own axes.
label_panels(fig, {ax_spectra: ("a)", -0.22), ax_correlations: ("b)", -0.22)},
             align=[(ax_spectra, ax_correlations)])

fig.savefig(ROOT / "figures" / "stimulus_overview_supplementary.pdf",
            bbox_inches="tight", pad_inches=0.3)
fig.savefig(ROOT / "figures" / "stimulus_overview_supplementary.png",
            bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

## Summary

**The set.** 676 listening runs were presented (plus 49 resting-state runs, ignored
here). 620 have an EEG file in the download, 617 became standardised trials, and
563 survive the condition and recording exclusions: 69 stimuli, 81 subjects,
134.7 listening hours. Of the 85 subjects presented, four contribute nothing at
all -- sub-019, sub-020, sub-021 and sub-026 -- because no run of theirs was
recorded. One recording, audiobook 1, accounts for 74 of the subjects.

**Unit rates.** Medians over stimuli: sentences 0.20 Hz, clauses 0.32 Hz,
intonation units 0.38 Hz, words 2.53 Hz, syllables 3.28 Hz by the heuristic and
3.40 Hz by de Jong, phonemes 8.87 Hz. The two syllable estimators correlate at
0.74, which is close enough to agree on the picture and far enough that they are
not interchangeable.

**One shared rate axis.** Across the four levels of the hierarchy -- intonation
units, words, syllables and phonemes -- all six pairs correlate significantly over
the 69 stimuli (Pearson, FDR < 0.05), from 0.50 to 0.82 with a median of 0.69. A
recording that runs fast at one level runs fast at the others, so these are not
four independent handles on the material: treat them as one speaking-rate axis
measured four ways. Median F0 is deliberately not in that matrix -- it is not a
unit rate -- but it is worth noting that it does not follow the axis either, which
is what makes pitch and rate separable in the coherence analysis.

**Modulation spectra.** The listening-time-weighted envelope spectrum peaks at
3.6 Hz by the `f * P(f)` criterion, and at 3.8 Hz once the longer silences are cut
-- which also flattens the sub-1 Hz shoulder that pauses were feeding. Silence is
11-35% of each stimulus (median 22%).

**Mastoids.** 21 of the 617 standardised recordings carry neither `A1` nor `A2`,
and none carries just one. All 21 are analysed trials, all in the natural
condition, and they belong to exactly three of the 81 subjects -- sub-052, sub-061
and sub-064 -- at seven trials each. Mastoids are a property of the capping rather
than of the run, so this is all-or-nothing per subject: no subject has some
recordings with them and some without. The linked-mastoid reference therefore runs
on 542 trials from 78 subjects, while the average and FCz references keep all 563.

## Coherence and the derivative spectrum, $f^2 P(f)$, over the full range

The envelope's modulation spectrum weighted by $f^2$: up to a constant, the power spectrum of
the envelope's *derivative*, so what drives how fast the envelope changes rather than where its
variance sits. That makes it the stimulus-side counterpart of the onset regressor in notebook 7,
which is that derivative half-wave rectified. It runs from the first non-DC bin to the Nyquist
frequency, for the whole stimuli with their pauses. It is drawn as per-stimulus traces with
the listening-time-weighted mean on top, and each spectrum is divided by its own total power.

Above it is the RMS EEG-envelope coherence and its pooled null, from notebook 2's cache, on the
same frequency bins.

In [ ]:
# ===================== coherence + f^2 P(f) options ==========================
F2_FIGSIZE = (5.6, 4.4)
F2_XTICKS = [1, 10, 100]        # majors at each decade
F2_YSCALE = "log"               # the curve spans several decades over the full range
# Notebook 2's grand average, read from its cache rather than re-pooled: the same RMS trace
# and pooled null its figure draws. Run notebook 2 first if the file is missing.
COHERENCE_REFERENCE = "avg"
COHERENCE_CACHE = ROOT / "figures" / "figure_data" / COHERENCE_REFERENCE / "grand_average_coherence.npz"
NULL_FILL = "#b9b9b4"           # notebook 2's null colour
# =============================================================================

if not COHERENCE_CACHE.exists():
    raise FileNotFoundError(f"{COHERENCE_CACHE} is missing; run 2_grand_average_coherence.ipynb "
                            f"with PRIMARY_REFERENCE = {COHERENCE_REFERENCE!r} first")
with np.load(COHERENCE_CACHE) as saved:
    coherence_frequencies = saved["frequencies"]
    # Complex-average coherence per channel -> RMS magnitude over the channels, as notebook 2.
    rms_coherence = np.sqrt((np.abs(saved["coherence"]) ** 2).mean(-1))
    coherence_null = saved["null_spectra"]
# Same 5 s Welch window at the same rate on both sides, so the two panels share bins.
assert np.allclose(coherence_frequencies, supp_frequencies), "frequency grids differ"

# f^2 P(f) is the power spectrum of the envelope's derivative, up to (2 pi)^2: what drives
# how fast the envelope changes rather than where its variance sits. It has no area meaning on
# a log axis, so each spectrum is divided by its own total power (the integral of P over every
# bin above DC) instead. Display scaling only; the Welch estimates are untouched.
full_band = supp_frequencies > 0
frequency_step = supp_frequencies[1] - supp_frequencies[0]
f2_frequencies = supp_frequencies[full_band]


def derivative_weighted(psd):
    """`f**2 * P(f)` over every non-DC bin, divided by the spectrum's total power."""
    psd = np.asarray(psd)[..., full_band]
    total = psd.sum(axis=-1, keepdims=True) * frequency_step
    return f2_frequencies ** 2 * psd / total


f2_values = derivative_weighted(raw_psds)
f2_average = derivative_weighted(np.average(raw_psds, axis=0, weights=WEIGHTS))

f2_fig, (ax_coherence, ax_f2) = plt.subplots(2, 1, figsize=F2_FIGSIZE, sharex=True,
                                             layout="constrained")
# Room between the panels for b)'s letter, which otherwise sits on a)'s x axis.
f2_fig.get_layout_engine().set(h_pad=0.12)

# --- a) RMS coherence over the range of the pooled null ----------------------------
ax_coherence.fill_between(f2_frequencies, coherence_null.min(0)[full_band],
                          coherence_null.max(0)[full_band], color=NULL_FILL, alpha=0.85, lw=0,
                          zorder=1, label="Noise level")
ax_coherence.plot(f2_frequencies, rms_coherence[full_band], color=INK, lw=0.8, zorder=2,
                  label="EEG-envelope coherence")
ax_coherence.set_yscale("log")
ax_coherence.set_ylabel("RMS coherence\nmagnitude")
ax_coherence.legend(loc="upper right", fontsize=TICK_FONTSIZE, frameon=False)
bare(ax_coherence)

# --- b) f^2 P(f), with pauses ----------------------------------------------------
ax_f2.plot(f2_frequencies, f2_values.T, color=INK, lw=0.4, alpha=0.16, zorder=1)
ax_f2.plot(f2_frequencies, f2_average, color=INK, lw=1.8, zorder=3)
ax_f2.set_yscale(F2_YSCALE)
ax_f2.set_ylabel(r"$f^2\,P(f)$" "\n" r"($\div$ total power)")
ax_f2.set_xlabel("Modulation frequency (Hz)")
bare(ax_f2)

ax_f2.set_xscale("log")
ax_f2.set_xlim(f2_frequencies[0], f2_frequencies[-1])
ax_f2.set_xticks(F2_XTICKS, [f"{tick:g}" for tick in F2_XTICKS])

label_panels(f2_fig, {ax_coherence: ("a)", -0.15), ax_f2: ("b)", -0.15)})

f2_fig.savefig(ROOT / "figures" / f"coherence_and_f2_spectrum_{COHERENCE_REFERENCE}.pdf",
               bbox_inches="tight", pad_inches=0.3)
f2_fig.savefig(ROOT / "figures" / f"coherence_and_f2_spectrum_{COHERENCE_REFERENCE}.png",
               bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

# Two peaks, because above ~80 Hz the curve is following voice pitch rather than speech rhythm,
# and that is where the global maximum lands.
F2_RHYTHM_BAND = (1.0, 30.0)
in_rhythm = (f2_frequencies >= F2_RHYTHM_BAND[0]) & (f2_frequencies <= F2_RHYTHM_BAND[1])
print(f"{COHERENCE_REFERENCE} reference coherence, {len(coherence_null)} null draws")
print(f"f^2 P(f) (with pauses) peaks at "
      f"{f2_frequencies[in_rhythm][np.argmax(f2_average[in_rhythm])]:.1f} Hz within "
      f"{F2_RHYTHM_BAND[0]:g}-{F2_RHYTHM_BAND[1]:g} Hz, "
      f"{f2_frequencies[np.argmax(f2_average)]:.0f} Hz overall")